In [1]:
import os

from deepfix_sdk import DeepFixClient

In [ ]:
os.environ["DEEPFIX_API_KEY"] = ... # get your API key at https://deepfix.delcaux.com

In [3]:
client = DeepFixClient(timeout=120)

## Classification

In [4]:
from deepfix_sdk.data.datasets import TabularDataset
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

In [5]:
# Load data
X, y = load_breast_cancer(as_frame=True, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
dataset_name = "breast_cancer_classification"

label = "target"
train = X_train.copy()
train[label] = y_train
cat_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

test = X_test.copy()
test[label] = y_test

In [6]:
train_data = TabularDataset(
    dataset=train, dataset_name=dataset_name, label=label, cat_features=cat_features
)
val_data = TabularDataset(
    dataset=test, dataset_name=dataset_name, label=label, cat_features=cat_features
)

In [7]:
#train_data.data.head()

In [8]:
# Fit model
model_name = "HistGradientBoostingClassifier"
clf = HistGradientBoostingClassifier(max_depth=3)
clf = clf.fit(train_data.X, train_data.y)

In [ ]:
result = client.get_diagnosis(
    train_data=train_data,
    test_data=val_data,
    model_name=model_name,
    model=clf,
    language="english",
)

In [13]:
result = client.diagnose(
    dataset_name=client.get_dataset_name(train_data, val_data),
    model_name=model_name,
    language="english",
)

Connecting to: https://deepfix.delcaux.com/api/v1/analyse

Output()

✓ Analysis complete!

In [14]:
# Visualize results
result.to_text(verbose=False)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ Integrated analyses reveal critical gaps (unfitted model, severe multicollinearity) blocking deployment and          │
│ validation, compounded by class imbalance, overfitting risks from high dim/outliers/leakage-prone features, and      │
│ reproducibility issues. Dataset strong but preprocessing essential (PCA/class_weight/robust scaling/CV). Prioritize  │
│ fitting model with fixes, then re-run checks for generalization.                                                     │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  8                                                            
 Severity Distribution           MEDIUM: 4  HIGH: 2  LOW: 2                                   

                                  HIGH Severity Issues (2)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Model artifacts lack fitted state,       │ Fit the model on training data,          │
│     │ making them unusable for inference,      │ serialize with joblib.dump(model,        │
│     │ deployment, or validation.               │ 'model.joblib'), and include key         │
│     │ Evidence: ModelCheckpoint: No serialized │ attributes (classes_, feature_names_in_, │
│     │ weights, classes_, n_iter_, scores; only │ n_iter_, train/validation scores) in     │
│     │ hyperparameters and docstring.           │ metadata.                                │
│     │ Deepchecks reports assume a model but    │ Essential for predictions and assessing  │
│     │ perfect AUC (0.99-1.0) unverified        │ Deepchecks metrics; hyperparameters      │
│     │ without fitted state.                    │ alone yield untrained model, risking     │
│     │                                          │ invalid analyses.                        │
│ 2   │ Severe multicollinearity among features  │ Apply PCA for dimensionality reduction   │
│     │ risks model instability and poor         │ or drop one feature per correlated pair  │
│     │ generalization.                          │ (e.g., retain 'mean' over 'worst');      │
│     │ Evidence: Deepchecks: >20 pairs corr>0.9 │ compute corr matrix first.               │
│     │ (e.g., mean/worst radius, perimeter,     │ Reduces variance inflation, effective    │
│     │ area); Dataset: Expected high corr       │ dim (30 feats), improves                 │
│     │ (radius/perimeter/area), no stats        │ interpretability; tree models tolerate   │
│     │ provided but high unique% suggests       │ but still benefits generalization on     │
│     │ redundancy.                              │ small n=455.                             │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (4)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Class imbalance (~63% malignant          │ Set class_weight='balanced' in           │
│     │ majority) biases model toward majority   │ HistGradientBoostingClassifier;          │
│     │ class.                                   │ alternatively, use SMOTE oversampling.   │
│     │ Evidence: Dataset: Target mean           │ Weights inverse to frequency, boosts     │
│     │ 0.626/0.632 train/test (~37% benign);    │ minority recall/F1; aligns Dataset/Model │
│     │ ModelCheckpoint: class_weight=null;      │ advice, prevents majority bias in cancer │
│     │ Deepchecks notes imbalance but no        │ diagnosis.                               │
│     │ explicit fail.                           │                                          │
│ 2   │ Near-perfect performance (AUC 0.99-1.0)  │ Perform k-fold CV post-fitting, add      │
│     │ suggests overfitting or data leakage.    │ regularization (max_depth tune, early    │
│     │ Evidence: Deepchecks: ROC perfect across │ stopping), monitor train-test gap; audit │
│     │ train/test but suspicious for tabular    │ high-PPS features (e.g., worst perimeter │
│     │ data; Dataset: High dim (30 feats, 15:1  │ 0.77).                                   │
│     │ ratio), outliers; Model unfitted         │ Validates generalization; high           │
│     │ prevents CV confirmation.                │ dim/outliers + perfect scores indicate   │
│     │                                          │ overfit risk; PPS>0.7 flags leakage      │
│     │                                          │ potential.                               │
│ 3   │ High predictive power in few features    │ Audit features for leakage (e.g.,        │
│     │ indicates potential leakage or           │ label-derived); use feature selection    │
│     │ redundancy.                              │ (mutual info, SelectFromModel on         │
│     │ Evidence: Deepchecks: 3 feats >0.7 PPS   │ importances).                            │
│     │ (worst perimeter/concave points/radius); │ Ensures robust signals; combines with    │
│     │ train-test diff low but suspicious with  │ corr reduction for cleaner feature set.  │
│     │ correlations.                            │                                          │
│ 4   │ Reproducibility issues due to            │ Set random_state=42; add model card with │
│     │ random_state=null and lack of metadata.  │ dataset info, final metrics, CV results. │
│     │ Evidence: ModelCheckpoint:               │ Enables reproducible runs and quality    │
│     │ random_state=null; no training           │ assessment; critical for deployment and  │
│     │ metrics/dataset details; Deepchecks      │ debugging.                               │
│     │ drifts=0 but unverified.                 │                                          │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                   LOW Severity Issues (2)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Heavy-tailed features and outliers may   │ Apply robust scaling                     │
│     │ distort model fitting.                   │ (QuantileTransformer) or clip at 99th    │
│     │ Evidence: Dataset: E.g., area_error max  │ percentile; add outlier detection in     │
│     │ 13x mean, high std/mean >0.5; Deepchecks │ Deepchecks.                              │
│     │ lacks outlier checks.                    │ Mitigates extreme influence on trees;    │
│     │                                          │ preserves rare cases (e.g., aggressive   │
│     │                                          │ tumors).                                 │
│ 2   │ Dataset integrity high: complete, no     │ Proceed to modeling after addressing     │
│     │ missing/low-info features.               │ above; minor: explicit cat features list │
│     │ Evidence: Dataset: Full counts, >76%     │ if any.                                  │
│     │ unique; Deepchecks integrity passes; no  │ Minimizes preprocessing; focus efforts   │
│     │ cats/low-card.                           │ on higher risks.                         │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''